In [22]:
import csv
import time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def scrape_ikman_houses_2026(total_pages=2):
    driver = uc.Chrome()
    base_url = "https://ikman.lk/en/ads/sri-lanka/houses-for-sale"
    all_ads = []

    try:
        for page in range(1, total_pages + 1):
            url = f"{base_url}?page={page}"
            driver.get(url)
            print(f"Scanning Page {page}...")

            # 1. WAIT: Manually click any CAPTCHA if it appears
            time.sleep(5) 

            # 2. WAIT for the container to exist
            try:
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.TAG_NAME, "article")) or 
                    EC.presence_of_element_located((By.CSS_SELECTOR, "li[data-testid='ad-card']"))
                )
            except:
                print("Could not find property cards. Scrolling to try and trigger load...")

            # 3. SCROLL to trigger "Lazy Load"
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(2)

            # 4. CAPTURE: Look for anything that looks like an Ad link
            # We look for all <a> tags that contain the price "Rs"
            links = driver.find_elements(By.TAG_NAME, "a")
            
            for link in links:
                try:
                    text = link.text
                    if "Rs" in text and "\n" in text:
                        lines = text.split("\n")
                        
                        # Data layout: Title is usually 1st, Price contains 'Rs'
                        title = lines[0]
                        price = next((line for line in lines if "Rs" in line), "N/A")
                        details = lines[1] # Usually the bottom line
                        
                        all_ads.append({
                            "Title": title,
                            "Price": price,
                            "Details": details,
                            "URL": link.get_attribute("href")
                        })
                except:
                    continue

            print(f"Found {len(all_ads)} potential items so far.")

    finally:
        if all_ads:
            # Remove duplicates (one ad often has multiple links)
            unique_ads = {v['Title'] + v['Price']: v for v in all_ads}.values()
            
            with open('ikman_houses_2026.csv', 'w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=["Title", "Price", "Details", "URL"])
                writer.writeheader()
                writer.writerows(unique_ads)
            print(f"Success! {len(unique_ads)} unique houses saved to CSV.")
        
        driver.quit()

if __name__ == "__main__":
    scrape_ikman_houses_2026(total_pages=10)

Scanning Page 1...
Could not find property cards. Scrolling to try and trigger load...
Found 28 potential items so far.
Scanning Page 2...
Could not find property cards. Scrolling to try and trigger load...
Found 56 potential items so far.
Scanning Page 3...
Could not find property cards. Scrolling to try and trigger load...
Found 84 potential items so far.
Scanning Page 4...
Could not find property cards. Scrolling to try and trigger load...
Found 112 potential items so far.
Scanning Page 5...
Could not find property cards. Scrolling to try and trigger load...
Found 140 potential items so far.
Scanning Page 6...
Could not find property cards. Scrolling to try and trigger load...
Found 168 potential items so far.
Scanning Page 7...
Could not find property cards. Scrolling to try and trigger load...
Found 196 potential items so far.
Scanning Page 8...
Could not find property cards. Scrolling to try and trigger load...
Found 224 potential items so far.
Scanning Page 9...
Could not find p

In [30]:
import pandas as pd
df= pd.read_csv('ikman_houses_2026.csv')
df.head()

,Title,Price,Details,URL
0,House for Sale in Gampaha,"Rs 3,900,000","Bedrooms: 3, Bathrooms: 1",https://ikman.lk/en/ad/house-for-sale-in-gampa...
1,QUICK SALE:B/New 4/3BR House For Sale Canterbu...,"Rs 32,000,000","Bedrooms: 4, Bathrooms: 3",https://ikman.lk/en/ad/quick-saleb-new-4-3br-h...
2,FEATURED,"Rs 41,000,000",+12,https://ikman.lk/en/ad/pdinci-novuu-nv-nivs-vi...
3,Hokandara Brand New Architecturally Disignd Tw...,"Rs 69,500,000","Bedrooms: 4, Bathrooms: 4",https://ikman.lk/en/ad/hokandara-brand-new-arc...
4,Bokundara Architecture Designed Super Luxury T...,"Rs 41,500,000","Bedrooms: 4, Bathrooms: 3",https://ikman.lk/en/ad/bokundara-architecture-...


In [31]:
mask = df['Details'].str.contains('Bedroom|Bathroom', case=False, na=False)
df_filtered = df[mask]
df_filtered.head()

,Title,Price,Details,URL
0,House for Sale in Gampaha,"Rs 3,900,000","Bedrooms: 3, Bathrooms: 1",https://ikman.lk/en/ad/house-for-sale-in-gampa...
1,QUICK SALE:B/New 4/3BR House For Sale Canterbu...,"Rs 32,000,000","Bedrooms: 4, Bathrooms: 3",https://ikman.lk/en/ad/quick-saleb-new-4-3br-h...
3,Hokandara Brand New Architecturally Disignd Tw...,"Rs 69,500,000","Bedrooms: 4, Bathrooms: 4",https://ikman.lk/en/ad/hokandara-brand-new-arc...
4,Bokundara Architecture Designed Super Luxury T...,"Rs 41,500,000","Bedrooms: 4, Bathrooms: 3",https://ikman.lk/en/ad/bokundara-architecture-...
5,Piliyandala Architecture Designed Super Luxury...,"Rs 37,500,000","Bedrooms: 5, Bathrooms: 3",https://ikman.lk/en/ad/piliyandala-architectur...


In [32]:
df_filtered["Details"].value_counts()

Details
Bedrooms: 3, Bathrooms: 2    66
Bedrooms: 4, Bathrooms: 3    49
Bedrooms: 4, Bathrooms: 4    20
Bedrooms: 3, Bathrooms: 3    19
Bedrooms: 5, Bathrooms: 3    18
Bedrooms: 2, Bathrooms: 1    11
Bedrooms: 5, Bathrooms: 4    10
Bedrooms: 4, Bathrooms: 2    10
Bedrooms: 5, Bathrooms: 5    10
Bedrooms: 3, Bathrooms: 1    10
Bedrooms: 6, Bathrooms: 4     4
Bedrooms: 5, Bathrooms: 2     4
Bedrooms: 5, Bathrooms: 6     4
Bedrooms: 2, Bathrooms: 2     3
Bedrooms: 6, Bathrooms: 5     3
Bedrooms: 5, Bathrooms: 1     2
Bedrooms: 4, Bathrooms: 1     2
Bedrooms: 4, Bathrooms: 5     2
Bedrooms: 6, Bathrooms: 3     2
Bedrooms: 7, Bathrooms: 7     1
Bedrooms: 7, Bathrooms: 4     1
Bedrooms: 7, Bathrooms: 5     1
Bedrooms: 2, Bathrooms: 3     1
Bedrooms: 6, Bathrooms: 2     1
Bedrooms: 8, Bathrooms: 8     1
Bedrooms: 7, Bathrooms: 6     1
Bedrooms: 8, Bathrooms: 4     1
Bedrooms: 5, Bathrooms: 7     1
Bedrooms: 7, Bathrooms: 3     1
Bedrooms: 9, Bathrooms: 8     1
Name: count, dtype: int64

In [33]:
df_filtered[['Bedrooms', 'Bathrooms']] = df_filtered['Details'].str.extract(
    r'Bedrooms:\s*(\d+).*?Bathrooms:\s*(\d+)', 
    expand=True
)

df_filtered.head()
df_filtered.drop(columns=['Details'], inplace=True)
df = df_filtered[['Title', 'Price', 'Bedrooms', 'Bathrooms', 'URL']]

C:\Users\user\AppData\Local\Temp\ipykernel_24160\1133796787.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered[['Bedrooms', 'Bathrooms']] = df_filtered['Details'].str.extract(
C:\Users\user\AppData\Local\Temp\ipykernel_24160\1133796787.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered[['Bedrooms', 'Bathrooms']] = df_filtered['Details'].str.extract(
C:\Users\user\AppData\Local\Temp\ipykernel_24160\1133796787.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a 

In [34]:
df.head()

,Title,Price,Bedrooms,Bathrooms,URL
0,House for Sale in Gampaha,"Rs 3,900,000",3,1,https://ikman.lk/en/ad/house-for-sale-in-gampa...
1,QUICK SALE:B/New 4/3BR House For Sale Canterbu...,"Rs 32,000,000",4,3,https://ikman.lk/en/ad/quick-saleb-new-4-3br-h...
3,Hokandara Brand New Architecturally Disignd Tw...,"Rs 69,500,000",4,4,https://ikman.lk/en/ad/hokandara-brand-new-arc...
4,Bokundara Architecture Designed Super Luxury T...,"Rs 41,500,000",4,3,https://ikman.lk/en/ad/bokundara-architecture-...
5,Piliyandala Architecture Designed Super Luxury...,"Rs 37,500,000",5,3,https://ikman.lk/en/ad/piliyandala-architectur...
